# Phase 7: Subgroup Performance Audit & Demographic Sensitivity Analysis

**Project:** Smart Student Retention & Early Warning System  
**Cutoff Horizon:** Day 35 ($t \le 35$)  
**Policy Stance:** Audit-Only Demographic Governance  

---

## 1. Core Ethical Principles & Evaluation Scope

1. **Exclusion from Predictors:** Demographic and socioeconomic variables (`gender`, `age_band`, `disability`, `imd_band`, `highest_education`, `region`) are strictly excluded from baseline features to prevent direct disparate treatment and algorithmic redlining.
2. **Continuous Subgroup Auditing:** Excluding demographic predictors does not guarantee zero proxy reliance. Models must be continuously audited for selection rate parity (disparate impact) and error-rate parity (FPR/FNR balance).
3. **Separation of Classifier Diagnostic vs Operational Queue Policy:** The classifier-level diagnostic evaluates raw model probabilities at a fixed validation threshold ($\theta = 0.25$). The operational queue audit evaluates the actual intervention policy: caseload queue inclusion under advisor capacity constraints ($K=500$ per presentation) scored with calibrated probabilities.
4. **Statistical Stability Safeguard:** Subgroups with $N < 50$ carry explicit statistical instability warnings to prevent over-interpreting noisy ratio estimates.
5. **Fairness Evaluation Scope:** Do not claim that the model has passed fairness merely because one metric is within tolerance. Disparity metrics provide operational transparency; observational differences reflect historical signal distributions and must not be interpreted as causal student deficiencies or proof of model perfection.


In [ ]:
import sys
from pathlib import Path

import pandas as pd


def find_project_root() -> Path:
    current = Path.cwd().resolve()
    for directory in [current, *current.parents]:
        if (directory / 'pyproject.toml').is_file():
            return directory
    return current.parent

PROJECT_ROOT = find_project_root()
if str(PROJECT_ROOT / 'src') not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT / 'src'))

from student_retention.decision_support.prioritize_students import (
    build_student_support_queue,
)
from student_retention.fairness.subgroup_audit import (
    audit_all_protected_attributes,
    audit_operational_support_queue,
    run_demographic_sensitivity_experiment,
)
from student_retention.features.feature_selection import select_primary_features
from student_retention.modeling.calibration import load_calibrated_model
from student_retention.modeling.predict import load_model, load_operational_threshold
from student_retention.modeling.splits import split_by_student_group
from student_retention.paths import (
    MODELS_BASELINE_DIR,
    PROCESSED_FEATURES_DIR,
)

print(f'Project root: {PROJECT_ROOT}')


In [ ]:
# Load Policy C feature matrix
features_path = PROCESSED_FEATURES_DIR / 'feature_matrix_policy_c.parquet'
df = pd.read_parquet(features_path)
print(f'Loaded feature matrix: {len(df)} rows across {len(df.columns)} columns')

# Load trained Random Forest model
rf_model = load_model(MODELS_BASELINE_DIR / 'random_forest.joblib')

# Split held-out test cohort on student ID (zero group leakage)
train_df, test_df = split_by_student_group(df, test_size=0.20, random_state=42, group_col='id_student')
clean_test_x = select_primary_features(test_df, include_target=False, include_identifiers=False)

# Load validated operational threshold strictly from training CV metadata (provenance consistency)
operational_threshold = load_operational_threshold('random_forest')
print(f'Operational threshold (from 5-fold training CV metadata): {operational_threshold:.4f}')

# Score test set at operational threshold (Classifier-Level Diagnostic)
test_probs = rf_model.predict_proba(clean_test_x)[:, 1]
test_preds = (test_probs >= operational_threshold).astype(int)

test_eval_df = test_df.copy()
test_eval_df['predicted_label'] = test_preds
test_eval_df['risk_probability'] = test_probs

# Run complete classifier-level subgroup disparity audit
classifier_audit_df = audit_all_protected_attributes(
    df=test_eval_df,
    audit_scope='classifier_level',
    decision_policy=f'fixed_threshold_{operational_threshold:.2f}',
)
print(f'Classifier audit completed: {len(classifier_audit_df)} subgroups audited across 6 protected attributes.')

# Inspect warnings across statistically stable subgroups
stable_groups = classifier_audit_df[classifier_audit_df['is_statistically_stable']]
di_warns = stable_groups[stable_groups['fairness_warning_flag'].str.contains('disparate_impact')]
fpr_warns = stable_groups[stable_groups['fairness_warning_flag'].str.contains('elevated_fpr')]
print(f'Classifier: Stable subgroups with Disparate Impact warnings (DIR outside [0.80, 1.25]): {len(di_warns)}')
print(f'Classifier: Stable subgroups with Elevated FPR warnings (FPR ratio > 1.50): {len(fpr_warns)}')

classifier_audit_df[['protected_attribute', 'subgroup', 'sample_size', 'base_rate', 'selection_rate', 'disparate_impact_ratio', 'fpr_ratio', 'fairness_warning_flag']].head(15)


In [ ]:
# Load Calibrated Model and Generate Operational Support Queue on Test Cohort
cal_path = MODELS_BASELINE_DIR / 'random_forest_calibrated.joblib'
calibrated_model = load_calibrated_model(cal_path)

queue_df = build_student_support_queue(
    feature_df=test_df,
    model=calibrated_model,
    urgent_threshold=0.70,
    watch_threshold=0.40,
    capacity_k=500,
)

# Compute Calibrated Risk Probabilities for Cohort
test_eval_queue_df = test_df.copy()
test_eval_queue_df['calibrated_risk_probability'] = calibrated_model.predict_proba(clean_test_x)[:, 1]

# Execute Operational Support Queue Subgroup Fairness Audit
queue_audit_df = audit_operational_support_queue(
    queue_df=queue_df,
    cohort_df=test_eval_queue_df,
    prob_col='calibrated_risk_probability',
)
print(f'Operational Queue Audit completed: {len(queue_audit_df)} subgroups audited across 6 protected attributes.')

q_stable = queue_audit_df[queue_audit_df['is_statistically_stable']]
q_di_warns = q_stable[q_stable['fairness_warning_flag'].str.contains('disparate_impact')]
q_fpr_warns = q_stable[q_stable['fairness_warning_flag'].str.contains('elevated_fpr')]
print(f'Queue: Stable subgroups with Disparate Impact warnings (DIR outside [0.80, 1.25]): {len(q_di_warns)}')
print(f'Queue: Stable subgroups with Elevated FPR warnings (FPR ratio > 1.50): {len(q_fpr_warns)}')

queue_audit_df[['protected_attribute', 'subgroup', 'sample_size', 'base_rate', 'selection_rate', 'disparate_impact_ratio', 'recall', 'fpr_ratio', 'fairness_warning_flag']].head(15)


In [ ]:
# Run Demographic Sensitivity Experiment (Primary Baseline vs Demographic-Aware Model)
sensitivity = run_demographic_sensitivity_experiment(
    feature_df=df,
    test_size=0.20,
    random_state=42,
    operational_threshold=operational_threshold,
)

print('\n--- Overall Model Performance Comparison ---')
print(sensitivity['model_comparison'].to_string(index=False))

print('\n--- Policy Findings ---')
print(sensitivity['summary_findings']['policy_recommendation'])


## 2. Key Audit Observations & Recommendations

1. **Classifier-Level vs Operational Queue Separation:**
   - **Classifier-Level Diagnostic:** At a fixed global decision threshold ($\theta = 0.25$), the uncalibrated model produces 14 disparate impact warnings and 7 elevated FPR ratio warnings, reflecting regional base-rate variations and score compression.
   - **Operational Support Queue Audit:** When evaluated under the actual calibrated queue allocation policy ($K=500$ capacity per academic presentation), disparate impact warnings decrease to 4 stable subgroups (`gender=F`, `age_band=35-55`, `region=North Western Region`, `region=Scotland`) and zero elevated FPR ratio warnings occur.
2. **Demographic Sensitivity Finding:** Incorporating demographic predictors yields negligible predictive gain while introducing direct disparate treatment and institutional redlining risks. Excluded baseline status is strictly justified.
3. **Pastoral Resource Allocation:** Where elevated alert rates occur (e.g. students declaring disabilities or with non-traditional educational backgrounds), outreach must strictly take the form of supportive pastoral and accessibility check-ins, rather than punitive academic tracking.
4. **Caseload Grain & Aggregation Notice:** The demonstration queue is evaluated at the course-enrollment grain (2,000 slots across 1,827 unique students). A true 500-unique-student operational queue remains unresolved pending institutional governance approval of a cross-module risk aggregation policy.
